# Camada Silver - Transformação e Padronização

Este notebook realiza o tratamento e a padronização dos dados armazenados na camada Bronze.

## Objetivos

- selecionar os dados necessários para o período de análise;
- corrigir tipos e formatos de dados;
- padronizar nomes e valores;
- tratar diferenças de granularidade entre as fontes;
- preparar os dados para integração e análise na camada Gold.

## Tabelas de origem

- `bronze_ibge_pam_soja`
- `bronze_anp_biodiesel`
- `bronze_anp_materia_prima`

In [0]:
# Leitura das tabelas da camada Bronze

df_ibge = spark.table("bronze_ibge_pam_soja")
df_biodiesel = spark.table("bronze_anp_biodiesel")
df_materia_prima = spark.table("bronze_anp_materia_prima")

print("Registros IBGE:", df_ibge.count())
print("Registros ANP - Biodiesel:", df_biodiesel.count())
print("Registros ANP - Matérias-primas:", df_materia_prima.count())

Registros IBGE: 1944
Registros ANP - Biodiesel: 23864
Registros ANP - Matérias-primas: 4745


In [0]:
# Inspeção das variáveis disponíveis nos dados do IBGE

df_ibge.select(
    "D2C",
    "D2N",
    "MC",
    "MN"
).distinct().orderBy("D2C").show(truncate=False)

+-------+-----------------------------------------------------------------+----+-----------------------+
|D2C    |D2N                                                              |MC  |MN                     |
+-------+-----------------------------------------------------------------+----+-----------------------+
|1000215|Valor da produção - percentual do total geral                    |2   |Percentual             |
|1000216|Área colhida - percentual do total geral                         |2   |Percentual             |
|1008331|Área plantada ou destinada à colheita - percentual do total geral|2   |Percentual             |
|112    |Rendimento médio da produção                                     |33  |Quilogramas por Hectare|
|214    |Quantidade produzida                                             |1017|Toneladas              |
|215    |Valor da produção                                                |40  |Mil Reais              |
|216    |Área colhida                                  

In [0]:
from pyspark.sql import functions as F

# Variáveis do IBGE relevantes para o MVP
variaveis_ibge = ["8331", "216", "214", "112", "215"]

df_ibge_filtrado = (
    df_ibge
    .filter(
        (F.col("D2C").isin(variaveis_ibge)) &
        (F.col("D3N").cast("int").between(2015, 2023))
    )
)

print("Registros após o filtro:", df_ibge_filtrado.count())

df_ibge_filtrado.select(
    "D1N", "D2C", "D2N", "D3N", "D4N", "V", "MN"
).show(20, truncate=False)

Registros após o filtro: 1215
+-------+---+----------------------------+----+--------------+-----+-----------------------+
|D1N    |D2C|D2N                         |D3N |D4N           |V    |MN                     |
+-------+---+----------------------------+----+--------------+-----+-----------------------+
|Alagoas|214|Quantidade produzida        |2015|Soja (em grão)|550  |Toneladas              |
|Alagoas|214|Quantidade produzida        |2016|Soja (em grão)|1043 |Toneladas              |
|Alagoas|214|Quantidade produzida        |2017|Soja (em grão)|1240 |Toneladas              |
|Alagoas|214|Quantidade produzida        |2018|Soja (em grão)|2475 |Toneladas              |
|Alagoas|214|Quantidade produzida        |2019|Soja (em grão)|10363|Toneladas              |
|Alagoas|214|Quantidade produzida        |2020|Soja (em grão)|4640 |Toneladas              |
|Alagoas|214|Quantidade produzida        |2021|Soja (em grão)|9391 |Toneladas              |
|Alagoas|214|Quantidade produzida       

In [0]:
# Transformação dos dados do IBGE para formato analítico

df_ibge_silver = (
    df_ibge_filtrado
    .groupBy("D1C", "D1N", "D3N")
    .pivot("D2C", variaveis_ibge)
    .agg(F.first("V"))
    .select(
        F.col("D1C").alias("codigo_uf"),
        F.col("D1N").alias("uf"),
        F.col("D3N").cast("int").alias("ano"),
        F.expr("try_cast(`8331` as double)").alias("area_plantada_ha"),
        F.expr("try_cast(`216` as double)").alias("area_colhida_ha"),
        F.expr("try_cast(`214` as double)").alias("producao_soja_t"),
        F.expr("try_cast(`112` as double)").alias("rendimento_kg_ha"),
        F.expr("try_cast(`215` as double)").alias("valor_producao_mil_reais")
    )
)

print("Registros Silver IBGE:", df_ibge_silver.count())

display(
    df_ibge_silver.orderBy("uf", "ano")
)

Registros Silver IBGE: 243


codigo_uf,uf,ano,area_plantada_ha,area_colhida_ha,producao_soja_t,rendimento_kg_ha,valor_producao_mil_reais
12,Acre,2015,null,null,null,null,null
12,Acre,2016,100.0,100.0,150.0,1500.0,195.0
12,Acre,2017,127.0,127.0,261.0,2055.0,277.0
12,Acre,2018,480.0,480.0,1410.0,2938.0,1739.0
12,Acre,2019,1660.0,1590.0,5051.0,3177.0,6360.0
12,Acre,2020,3280.0,3280.0,10380.0,3165.0,14485.0
12,Acre,2021,6185.0,5985.0,20156.0,3368.0,51934.0
12,Acre,2022,6570.0,6570.0,22667.0,3450.0,66396.0
12,Acre,2023,12010.0,12010.0,45732.0,3808.0,99681.0
27,Alagoas,2015,278.0,278.0,550.0,1978.0,738.0


In [0]:
# Verificação de valores nulos após a transformação

colunas_numericas = [
    "area_plantada_ha",
    "area_colhida_ha",
    "producao_soja_t",
    "rendimento_kg_ha",
    "valor_producao_mil_reais"
]

df_ibge_silver.select(
    *[
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in colunas_numericas
    ]
).show()

+----------------+---------------+---------------+----------------+------------------------+
|area_plantada_ha|area_colhida_ha|producao_soja_t|rendimento_kg_ha|valor_producao_mil_reais|
+----------------+---------------+---------------+----------------+------------------------+
|              54|             55|             55|              55|                      55|
+----------------+---------------+---------------+----------------+------------------------+



In [0]:
# Investigação dos registros com valores ausentes

df_ibge_nulos = (
    df_ibge_silver
    .filter(
        F.col("producao_soja_t").isNull()
    )
    .select(
        "codigo_uf",
        "uf",
        "ano",
        "area_plantada_ha",
        "area_colhida_ha",
        "producao_soja_t",
        "rendimento_kg_ha",
        "valor_producao_mil_reais"
    )
    .orderBy("uf", "ano")
)

print("Registros sem informação de produção:", df_ibge_nulos.count())

display(df_ibge_nulos)

Registros sem informação de produção: 55


codigo_uf,uf,ano,area_plantada_ha,area_colhida_ha,producao_soja_t,rendimento_kg_ha,valor_producao_mil_reais
12,Acre,2015,null,null,null,null,null
13,Amazonas,2015,null,null,null,null,null
13,Amazonas,2016,null,null,null,null,null
13,Amazonas,2017,null,null,null,null,null
13,Amazonas,2018,null,null,null,null,null
23,Ceará,2015,null,null,null,null,null
23,Ceará,2016,null,null,null,null,null
23,Ceará,2017,null,null,null,null,null
23,Ceará,2019,null,null,null,null,null
32,Espírito Santo,2015,null,null,null,null,null


### Tratamento de valores ausentes - IBGE

Durante a transformação dos dados do IBGE, valores não numéricos presentes na fonte foram convertidos para `NULL`.

A análise dos registros resultantes mostrou que os valores ausentes estão concentrados em determinadas combinações de Unidade da Federação e ano, frequentemente afetando simultaneamente as variáveis de área, produção, rendimento e valor da produção.

Optou-se por preservar esses valores como `NULL`, em vez de substituí-los por zero, pois a ausência de informação na fonte não implica necessariamente inexistência de produção. Esse tratamento evita introduzir valores artificiais no conjunto de dados.

In [0]:
# Mapeamento das Unidades da Federação para as grandes regiões

regioes = {
    "Acre": "NORTE",
    "Amapá": "NORTE",
    "Amazonas": "NORTE",
    "Pará": "NORTE",
    "Rondônia": "NORTE",
    "Roraima": "NORTE",
    "Tocantins": "NORTE",

    "Alagoas": "NORDESTE",
    "Bahia": "NORDESTE",
    "Ceará": "NORDESTE",
    "Maranhão": "NORDESTE",
    "Paraíba": "NORDESTE",
    "Pernambuco": "NORDESTE",
    "Piauí": "NORDESTE",
    "Rio Grande do Norte": "NORDESTE",
    "Sergipe": "NORDESTE",

    "Distrito Federal": "CENTRO-OESTE",
    "Goiás": "CENTRO-OESTE",
    "Mato Grosso": "CENTRO-OESTE",
    "Mato Grosso do Sul": "CENTRO-OESTE",

    "Espírito Santo": "SUDESTE",
    "Minas Gerais": "SUDESTE",
    "Rio de Janeiro": "SUDESTE",
    "São Paulo": "SUDESTE",

    "Paraná": "SUL",
    "Rio Grande do Sul": "SUL",
    "Santa Catarina": "SUL"
}

mapa_regioes = F.create_map(
    *[F.lit(x) for item in regioes.items() for x in item]
)

df_ibge_silver = (
    df_ibge_silver
    .withColumn("regiao", mapa_regioes[F.col("uf")])
)

display(
    df_ibge_silver
    .select("codigo_uf", "uf", "regiao", "ano", "producao_soja_t")
    .orderBy("uf", "ano")
)

codigo_uf,uf,regiao,ano,producao_soja_t
12,Acre,NORTE,2015,null
12,Acre,NORTE,2016,150.0
12,Acre,NORTE,2017,261.0
12,Acre,NORTE,2018,1410.0
12,Acre,NORTE,2019,5051.0
12,Acre,NORTE,2020,10380.0
12,Acre,NORTE,2021,20156.0
12,Acre,NORTE,2022,22667.0
12,Acre,NORTE,2023,45732.0
27,Alagoas,NORDESTE,2015,550.0


### Padronização regional

As Unidades da Federação foram associadas às cinco grandes regiões brasileiras (Norte, Nordeste, Centro-Oeste, Sudeste e Sul).

A validação realizada após o mapeamento não identificou registros sem região associada, permitindo utilizar a dimensão regional posteriormente na integração dos dados do IBGE com os dados da ANP.

In [0]:
# Validação do mapeamento de regiões

df_ibge_silver.filter(
    F.col("regiao").isNull()
).select(
    "codigo_uf",
    "uf"
).distinct().show(truncate=False)

+---------+---+
|codigo_uf|uf |
+---------+---+
+---------+---+



In [0]:
# Inspeção da estrutura dos dados de produção de biodiesel da ANP

df_biodiesel.printSchema()

display(
    df_biodiesel.limit(20)
)

root
 |-- ano: long (nullable = true)
 |-- mes: string (nullable = true)
 |-- grande_regiao: string (nullable = true)
 |-- unidade_federacao: string (nullable = true)
 |-- produtor: string (nullable = true)
 |-- produto: string (nullable = true)
 |-- producao: string (nullable = true)



ano,mes,grande_regiao,unidade_federacao,produtor,produto,producao
2016,SET,REGIÃO CENTRO-OESTE,MATO GROSSO,BIO VIDA,BIODIESEL,0
2016,AGO,REGIÃO CENTRO-OESTE,MATO GROSSO,BIO VIDA,BIODIESEL,0
2016,JUL,REGIÃO CENTRO-OESTE,MATO GROSSO,BIO VIDA,BIODIESEL,0
2016,JUN,REGIÃO CENTRO-OESTE,MATO GROSSO,BIO VIDA,BIODIESEL,0
2016,MAI,REGIÃO CENTRO-OESTE,MATO GROSSO,BIO VIDA,BIODIESEL,0
2016,ABR,REGIÃO CENTRO-OESTE,MATO GROSSO,BIO VIDA,BIODIESEL,0
2016,NOV,REGIÃO CENTRO-OESTE,MATO GROSSO,BIO VIDA,BIODIESEL,0
2016,AGO,REGIÃO SUL,RIO GRANDE DO SUL,BOCCHI,BIODIESEL,"3720,96"
2016,JUN,REGIÃO NORTE,TOCANTINS,BIOTINS,BIODIESEL,0
2016,ABR,REGIÃO NORTE,TOCANTINS,BIOTINS,BIODIESEL,0


In [0]:
# Inspeção dos valores categóricos da ANP

print("Grandes regiões:")
df_biodiesel.select("grande_regiao").distinct().orderBy("grande_regiao").show(
    truncate=False
)

print("Unidades da Federação:")
df_biodiesel.select("unidade_federacao").distinct().orderBy(
    "unidade_federacao"
).show(
    50,
    truncate=False
)

print("Produtos:")
df_biodiesel.select("produto").distinct().show(truncate=False)

Grandes regiões:
+--------------------+
|grande_regiao       |
+--------------------+
|REGIÃO CENTRO-OESTE|
|REGIÃO NORDESTE    |
|REGIÃO NORTE       |
|REGIÃO SUDESTE     |
|REGIÃO SUL         |
+--------------------+

Unidades da Federação:
+-------------------+
|unidade_federacao  |
+-------------------+
|BAHIA              |
|CEARÃ             |
|GOIÃS             |
|MARANHÃO          |
|MATO GROSSO        |
|MATO GROSSO DO SUL |
|MINAS GERAIS       |
|PARANÃ            |
|PARÃ              |
|PIAUÃ             |
|RIO DE JANEIRO     |
|RIO GRANDE DO NORTE|
|RIO GRANDE DO SUL  |
|RONDÃNIA          |
|SANTA CATARINA     |
|SÃO PAULO         |
|TOCANTINS          |
+-------------------+

Produtos:
+---------+
|produto  |
+---------+
|BIODIESEL|
+---------+



In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# Função para corrigir textos UTF-8 interpretados incorretamente como Latin-1
@F.udf(StringType())
def corrigir_encoding(texto):
    if texto is None:
        return None

    try:
        return texto.encode("latin1").decode("utf8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return texto


# Padronização dos dados de produção de biodiesel da ANP
df_biodiesel_silver = (
    df_biodiesel
    .filter(F.col("ano").between(2015, 2023))
    .withColumn("grande_regiao", corrigir_encoding(F.col("grande_regiao")))
    .withColumn("uf", corrigir_encoding(F.col("unidade_federacao")))
    .withColumn("produtor", corrigir_encoding(F.col("produtor")))
    .withColumn(
        "regiao",
        F.regexp_replace(F.col("grande_regiao"), "^REGIÃO ", "")
    )
    .withColumn(
        "producao_biodiesel",
        F.regexp_replace(F.col("producao"), ",", ".").cast("double")
    )
    .select(
        "ano",
        "mes",
        "regiao",
        "uf",
        "produtor",
        "produto",
        "producao_biodiesel"
    )
)

print("Registros Silver ANP - Biodiesel:", df_biodiesel_silver.count())

display(
    df_biodiesel_silver
    .orderBy("ano", "mes", "regiao", "uf")
    .limit(30)
)

Registros Silver ANP - Biodiesel: 11340


ano,mes,regiao,uf,produtor,produto,producao_biodiesel
2015,ABR,CENTRO-OESTE,GOIÁS,OLFAR,BIODIESEL,0.0
2015,ABR,CENTRO-OESTE,GOIÁS,BIONORTE,BIODIESEL,0.0
2015,ABR,CENTRO-OESTE,GOIÁS,MINERVA,BIODIESEL,771.639
2015,ABR,CENTRO-OESTE,GOIÁS,CARAMURU (SÃO SIMÃO),BIODIESEL,11106.68
2015,ABR,CENTRO-OESTE,GOIÁS,CEREAL,BIODIESEL,0.0
2015,ABR,CENTRO-OESTE,GOIÁS,BIONASA,BIODIESEL,0.0
2015,ABR,CENTRO-OESTE,GOIÁS,JATAI,BIODIESEL,0.0
2015,ABR,CENTRO-OESTE,GOIÁS,BINATURAL,BIODIESEL,8505.216
2015,ABR,CENTRO-OESTE,GOIÁS,CARAMURU (IPAMERI),BIODIESEL,8605.538
2015,ABR,CENTRO-OESTE,GOIÁS,GRANOL (ANÁPOLIS),BIODIESEL,24581.463


In [0]:
# Inspeção da estrutura dos dados de matérias-primas da ANP

df_materia_prima.printSchema()

display(
    df_materia_prima
    .orderBy("mes_ano", "regiao", "estado")
    .limit(30)
)

root
 |-- mes_ano: string (nullable = true)
 |-- regiao: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- produto: string (nullable = true)
 |-- quantidade_m3: long (nullable = true)



mes_ano,regiao,estado,produto,quantidade_m3
01/2017,CENTRO OESTE,GoiÃ¡s,GORDURA DE PORCO,102
01/2017,CENTRO OESTE,GoiÃ¡s,ÃLEO DE SOJA (GLYCINE MAX),29031
01/2017,CENTRO OESTE,GoiÃ¡s,OUTROS MATERIAIS GRAXOS,7520
01/2017,CENTRO OESTE,GoiÃ¡s,GORDURA BOVINA,3263
01/2017,CENTRO OESTE,Mato Grosso,ÃLEO DE FRITURA USADO,132
01/2017,CENTRO OESTE,Mato Grosso,GORDURA DE FRANGO,25
01/2017,CENTRO OESTE,Mato Grosso,OUTROS MATERIAIS GRAXOS,14729
01/2017,CENTRO OESTE,Mato Grosso,ÃLEO DE SOJA (GLYCINE MAX),32072
01/2017,CENTRO OESTE,Mato Grosso,GORDURA BOVINA,386
01/2017,CENTRO OESTE,Mato Grosso do Sul,GORDURA BOVINA,920


In [0]:
# Inspeção das matérias-primas disponíveis

df_materia_prima.select("produto") \
    .distinct() \
    .orderBy("produto") \
    .show(100, truncate=False)

+----------------------------------------------------+
|produto                                             |
+----------------------------------------------------+
|GORDURA BOVINA                                      |
|GORDURA DE FRANGO                                   |
|GORDURA DE PORCO                                    |
|OUTROS MATERIAIS GRAXOS                             |
|ÃCIDO GRAXO DE ÃLEO DE PALMA / DENDÃ             |
|ÃCIDO GRAXO DE ÃLEO DE SOJA                       |
|ÃLEO DE ALGODÃO (GOSSYPIUM HIRSUT)                |
|ÃLEO DE COLZA/CANOLA (BRESSICA CAMPESTRIS)         |
|ÃLEO DE FRITURA USADO                              |
|ÃLEO DE GIRASSOL (HELLANTHUS ANNUS)                |
|ÃLEO DE MILHO                                      |
|ÃLEO DE PALMA/DENDÃ (ELAEIS GUINEENSIS OU ELAEIS O|
|ÃLEO DE PALMISTE                                   |
|ÃLEO DE SOJA (GLYCINE MAX)                         |
+----------------------------------------------------+



In [0]:
# Padronização dos dados de matérias-primas da ANP

df_materia_prima_silver = (
    df_materia_prima
    .withColumn("estado", corrigir_encoding(F.col("estado")))
    .withColumn("produto", corrigir_encoding(F.col("produto")))
    .withColumn(
        "data_competencia",
        F.to_date(F.col("mes_ano"), "MM/yyyy")
    )
    .withColumn("ano", F.year(F.col("data_competencia")))
    .withColumn("mes", F.month(F.col("data_competencia")))
    .withColumn(
        "regiao",
        F.regexp_replace(
            F.upper(F.trim(F.col("regiao"))),
            r"\s+",
            "-"
        )
    )
    .withColumn(
        "quantidade_m3",
        F.col("quantidade_m3").cast("double")
    )
    .filter(F.col("ano").between(2015, 2023))
    .select(
        "data_competencia",
        "ano",
        "mes",
        "regiao",
        "estado",
        "produto",
        "quantidade_m3"
    )
)

print(
    "Registros Silver ANP - Matérias-primas:",
    df_materia_prima_silver.count()
)

display(
    df_materia_prima_silver
    .orderBy("data_competencia", "regiao", "estado", "produto")
    .limit(30)
)

Registros Silver ANP - Matérias-primas: 4745


data_competencia,ano,mes,regiao,estado,produto,quantidade_m3
2017-01-01,2017,1,CENTRO-OESTE,Goiás,GORDURA BOVINA,3263.0
2017-01-01,2017,1,CENTRO-OESTE,Goiás,GORDURA DE PORCO,102.0
2017-01-01,2017,1,CENTRO-OESTE,Goiás,OUTROS MATERIAIS GRAXOS,7520.0
2017-01-01,2017,1,CENTRO-OESTE,Goiás,ÓLEO DE SOJA (GLYCINE MAX),29031.0
2017-01-01,2017,1,CENTRO-OESTE,Mato Grosso,GORDURA BOVINA,386.0
2017-01-01,2017,1,CENTRO-OESTE,Mato Grosso,GORDURA DE FRANGO,25.0
2017-01-01,2017,1,CENTRO-OESTE,Mato Grosso,OUTROS MATERIAIS GRAXOS,14729.0
2017-01-01,2017,1,CENTRO-OESTE,Mato Grosso,ÓLEO DE FRITURA USADO,132.0
2017-01-01,2017,1,CENTRO-OESTE,Mato Grosso,ÓLEO DE SOJA (GLYCINE MAX),32072.0
2017-01-01,2017,1,CENTRO-OESTE,Mato Grosso do Sul,GORDURA BOVINA,920.0


In [0]:
# Validação das matérias-primas relacionadas à soja

df_materia_prima_silver \
    .filter(F.upper(F.col("produto")).contains("SOJA")) \
    .select("produto") \
    .distinct() \
    .orderBy("produto") \
    .show(truncate=False)

+---------------------------+
|produto                    |
+---------------------------+
|ÁCIDO GRAXO DE ÓLEO DE SOJA|
|ÓLEO DE SOJA (GLYCINE MAX) |
+---------------------------+



In [0]:
# Identificação do catálogo e schema atuais

spark.sql("""
    SELECT
        current_catalog() AS catalogo_atual,
        current_schema() AS schema_atual
""").show(truncate=False)

+--------------+------------+
|catalogo_atual|schema_atual|
+--------------+------------+
|workspace     |default     |
+--------------+------------+



In [0]:
# Persistência das tabelas da camada Silver

df_ibge_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_ibge_soja")

df_biodiesel_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_anp_biodiesel")

df_materia_prima_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_anp_materia_prima")

print("Tabelas Silver persistidas com sucesso.")

Tabelas Silver persistidas com sucesso.


In [0]:
# Validação das tabelas persistidas na camada Silver

spark.sql("""
    SELECT 'silver_ibge_soja' AS tabela, COUNT(*) AS registros
    FROM silver_ibge_soja

    UNION ALL

    SELECT 'silver_anp_biodiesel', COUNT(*)
    FROM silver_anp_biodiesel

    UNION ALL

    SELECT 'silver_anp_materia_prima', COUNT(*)
    FROM silver_anp_materia_prima
""").show(truncate=False)

+------------------------+---------+
|tabela                  |registros|
+------------------------+---------+
|silver_ibge_soja        |243      |
|silver_anp_biodiesel    |11340    |
|silver_anp_materia_prima|4745     |
+------------------------+---------+



In [0]:
# Schemas das tabelas Silver

spark.table("silver_ibge_soja").printSchema()
spark.table("silver_anp_biodiesel").printSchema()
spark.table("silver_anp_materia_prima").printSchema()

root
 |-- codigo_uf: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- ano: integer (nullable = true)
 |-- area_plantada_ha: double (nullable = true)
 |-- area_colhida_ha: double (nullable = true)
 |-- producao_soja_t: double (nullable = true)
 |-- rendimento_kg_ha: double (nullable = true)
 |-- valor_producao_mil_reais: double (nullable = true)
 |-- regiao: string (nullable = true)

root
 |-- ano: long (nullable = true)
 |-- mes: string (nullable = true)
 |-- regiao: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- produtor: string (nullable = true)
 |-- produto: string (nullable = true)
 |-- producao_biodiesel: double (nullable = true)

root
 |-- data_competencia: date (nullable = true)
 |-- ano: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- regiao: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- produto: string (nullable = true)
 |-- quantidade_m3: double (nullable = true)

